# Rust 123: Macros — Practice

Twin of `26_macros_e.ipynb`. The same desk, your hands on the wheel: every
exercise ships a complete desk program — macros **given**, its `main` (or a
helper fn) stubbed to `todo!()` — and you fill the body so the receipts match.
One kernel caveat drives the shape: macros defined in kernel cells are only
reliably invokable in statement position, so all macros here live **inside the
desk strings**, where plain rustc sees them.


### Contents

- **0.** How to complete these exercises — the desk (`0.1`, given), first call (`0.2`)
- **1.** Invocation shapes — delimiters (`1.1`), repetition (`1.2`)
- **2.** Fragments and the running desk build (`2.1`, `2.2`)
- **3.** Arms — routing (`3.1`), the no-arm error (`3.2`)
- **5.** Hygiene — the inner name stays in (`5.1`)
- **6.** Textual order — definition before call (`6.1`)
- **8.** Capstone — the three-macro build (`8.1`)


## 0. How to Complete These Exercises

Each exercise gives you a `const ... : &str` desk program — macros included,
bodies stubbed to `todo!()` — plus an immutable `// --- check ---` block that
compiles and runs it.

- **Replace every `todo!()`** so the check block's assertions hold. The doc
  comments and the section prose are the contract.
- The stubs are `fn`s (often `main`) *inside* the raw string — the same
  convention as chapter 25's suites.
- Stub state exits 101 at the first check; stub-state warnings (a given macro
  not yet called by your body) are normal until you fill the bodies in.
- Desk sources live in `scratch26/`; the last cell removes it.


In [ ]:
// 0.1 Given: the desk, same helpers as the example notebook.
use std::fs;
use std::process::Command;

/// Run the compiled desk program `name` with `args`; return (stdout, code).
fn desk(name: &str, args: &[&str]) -> (String, i32) {
    let run = Command::new(format!("./scratch26/desk_{name}.exe"))
        .args(args)
        .output()
        .unwrap();
    let out = String::from_utf8_lossy(&run.stdout).into_owned();
    let code = run.status.code().unwrap_or(-1);
    (out, code)
}

/// Write `src` to scratch26/desk_<name>.rs and compile it; return stderr.
/// Empty string = clean build. Errors and warnings are both data here.
fn desk_stderr(name: &str, src: &str) -> String {
    fs::create_dir_all("scratch26").unwrap();
    let rs = format!("scratch26/desk_{name}.rs");
    let exe = format!("scratch26/desk_{name}.exe");
    fs::write(&rs, src).unwrap();
    let out = Command::new("rustc")
        .args(["--edition", "2024", "-o", &exe, &rs])
        .output()
        .unwrap();
    String::from_utf8_lossy(&out.stderr).into_owned()
}

/// Compile `src` and run it with `args`: the usual two-step. Panics on any
/// compiler stderr -- warning-carrying sources go through `desk_stderr`.
fn desk_run(name: &str, src: &str, args: &[&str]) -> (String, i32) {
    let stderr = desk_stderr(name, src);
    assert!(
        stderr.is_empty(),
        "desk_{name} failed to compile:\n{stderr}"
    );
    desk(name, args)
}

println!("desk helpers ready (desk, desk_stderr, desk_run)");


### Exercise 0.2 — First call

`double!` is given inside the desk string; its `run` helper is stubbed. Make
it print `10`.


In [ ]:
// 0.2 Exercise: invoke the given macro from the desk's helper fn.
const DOUBLE_SRC: &str = r#"macro_rules! double {
    ($x:expr) => {
        $x * 2
    };
}

fn run_double() {
    todo!()
}

fn main() {
    run_double();
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("double26p", DOUBLE_SRC, &[]);
assert_eq!(out.trim(), "10", "out: {out:?}");
assert_eq!(code, 0);
println!("Exercise 0.2 passed");


## 1. Invocation Shapes

A macro can be invoked with `()`, `[]` or `{}` — but the *arm* decides which
shape it accepts. `pick!` has three arms; drive all three from the desk.


In [ ]:
// 1.1 Exercise: three arms, three delimiters, one joined receipt.
const PICK_SRC: &str = r#"macro_rules! pick {
    () => {
        "none".to_string()
    };
    [$x:expr] => {
        format!("one:{x}", x = $x)
    };
    [$x:expr, $y:expr] => {
        format!("two:{x}+{y}", x = $x, y = $y)
    };
}

fn run_pick() {
    todo!()
}

fn main() {
    run_pick();
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("pick26p", PICK_SRC, &[]);
assert_eq!(out.trim(), "none | one:6 | two:6+7", "out: {out:?}");
assert_eq!(code, 0);
println!("Exercise 1.1 passed");


### Exercise 1.2 — Repetition

`sum!` accepts zero or more comma-separated expressions (trailing commas
tolerated). Print both tallies: `sum3=24` then `sum0=0`.


In [ ]:
// 1.2 Exercise: zero or more, comma-separated.
const SUM_SRC: &str = r#"macro_rules! sum {
    ($($n:expr),* $(,)?) => {{
        #[allow(unused_mut)] // the empty call never mutates
        let mut acc = 0_i64;
        $( acc += $n; )*
        acc
    }};
}

fn run_sum() {
    todo!()
}

fn main() {
    run_sum();
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("sum26p", SUM_SRC, &[]);
let lines: Vec<String> = out.lines().map(String::from).collect();
assert_eq!(lines, vec!["sum3=24", "sum0=0"], "out: {out:?}");
assert_eq!(code, 0);
println!("Exercise 1.2 passed");


## 2. Fragments and the Running Desk Build

`ECHO_SRC` is the example notebook's running build (§2.1 there): one
`macro_rules!` whose `mature` literal arm returns `1.0` and whose `:tt`
catch-all returns `0.0`. It is **given**; drive it from your stub.


In [ ]:
// 2.0 Given: the running desk build's source.
const ECHO_SRC: &str = r#"macro_rules! special_returns {
    (mature) => {
        1.0_f64
    };
    ($other:tt) => {
        0.0_f64
    };
}

fn main() {
    let arg: Vec<String> = std::env::args().skip(1).collect();
    let cmd = arg.first().map(String::as_str).unwrap_or("");
    match cmd {
        "special_returns" => println!("{}", special_returns!(mature)),
        "any_token" => println!("{}", special_returns!(whatever)),
        _ => println!("unknown command"),
    }
}
"#;

let (_, code) = desk_run("echo26p", ECHO_SRC, &["special_returns"]);
assert_eq!(code, 0);
println!("desk build ready (scratch26/desk_echo26p.exe)");


### Exercise 2.1 — Drive the build

Run the compiled build on the `special_returns` command and hand back its
stdout.


In [ ]:
// 2.1 Exercise: one desk_run away.
/// Run desk_echo26p with ["special_returns"] and return its stdout as an
/// owned String.
fn special_returns_stdout() -> String {
    todo!()
}

// --- check (do not modify) ---
let got = special_returns_stdout();
assert_eq!(got.trim(), "1", "got: {got:?}");
println!("Exercise 2.1 passed");


### Exercise 2.2 — Fragments at work

`binding!` makes real bindings via `$name:ident`; `kv_map!` builds a
`HashMap` from `k => v` pairs. Print `qty=5` then `len=2`.


In [ ]:
// 2.2 Exercise: ident bindings + => pairs.
const FRAG_SRC: &str = r#"macro_rules! binding {
    ($name:ident = $val:expr) => {
        let $name = $val;
    };
}

macro_rules! kv_map {
    ($($k:expr => $v:expr),* $(,)?) => {{
        let mut m = std::collections::HashMap::new();
        $( m.insert($k, $v); )*
        m
    }};
}

fn run_frag() {
    todo!()
}

fn main() {
    run_frag();
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("frag26p", FRAG_SRC, &[]);
let lines: Vec<String> = out.lines().map(String::from).collect();
assert_eq!(lines, vec!["qty=5", "len=2"], "out: {out:?}");
assert_eq!(code, 0);
println!("Exercise 2.2 passed");


## 3. Arms: Routing and the No-Arm Error

Arms are tried top to bottom. `route!` routes two literal-token arms and has
no catch-all — which makes the *wrong* input a compile error worth reading as
data (§3.2).


In [ ]:
// 3.1 Exercise: route all three shapes.
const ROUTE_SRC: &str = r#"macro_rules! route {
    (buy $qty:expr) => {
        format!("BUY x{}", $qty)
    };
    (sell $qty:expr) => {
        format!("SELL x{}", $qty)
    };
    (hold) => {
        "HOLD".to_string()
    };
}

fn run_route() {
    todo!()
}

fn main() {
    run_route();
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("route26p", ROUTE_SRC, &[]);
assert_eq!(out.trim(), "BUY x5 | SELL x3 | HOLD", "out: {out:?}");
assert_eq!(code, 0);
println!("Exercise 3.1 passed");


### Exercise 3.2 — The no-arm error, captured

`NOARM_SRC` (given below, above the check) calls `double!(1, 2, 3)` — no arm
matches. Compile it on the desk and return the compiler's stderr.


In [ ]:
// 3.2 Exercise: the compile error as data.
const NOARM_SRC: &str = r#"macro_rules! double {
    ($x:expr) => {
        $x * 2
    };
}

fn main() {
    let _ = double!(1, 2, 3);
}
"#;

/// Compile NOARM_SRC via desk_stderr("noarm26p", ..) and return the stderr.
fn noarm_stderr() -> String {
    todo!()
}

// --- check (do not modify) ---
let stderr = noarm_stderr();
assert!(stderr.contains("no rules expected this token in macro call"), "stderr: {stderr}");
assert!(stderr.contains("while trying to match meta-variable `$x:expr`"), "stderr: {stderr}");
println!("Exercise 3.2 passed");


## 5. Hygiene: the Inner Name Stays In

`using_a!` declares `let a = 42;` *inside itself*. After the call, your own
`a` is untouched — the macro's binding is invisible and unused. Print the
caller's `a` from the desk.


In [ ]:
// 5.1 Exercise: prove hygiene wins.
const HYG_SRC: &str = r#"macro_rules! using_a {
    () => {
        #[allow(unused_variables)] // the macro's a dies on purpose
        let a = 42;
    };
}

fn run_hyg() {
    todo!()
}

fn main() {
    run_hyg();
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("hyg26p", HYG_SRC, &[]);
assert_eq!(out.trim(), "9", "out: {out:?}");
assert_eq!(code, 0);
println!("Exercise 5.1 passed");


## 6. Textual Order: Definition Before Call

`ORDER_SRC` (given, above the check) calls `double!` *above* its definition —
functions tolerate that; macros do not. Capture the error.


In [ ]:
// 6.1 Exercise: the order error as data.
const ORDER_SRC: &str = r#"fn main() {
    println!("{}", double!(3));
}

macro_rules! double {
    ($x:expr) => {
        $x * 2
    };
}
"#;

/// Compile ORDER_SRC via desk_stderr("order26p", ..) and return the stderr.
fn order_stderr() -> String {
    todo!()
}

// --- check (do not modify) ---
let stderr = order_stderr();
assert!(stderr.contains("cannot find macro `double` in this scope"), "stderr: {stderr}");
assert!(stderr.contains("consider moving the definition of `double` before this call"), "stderr: {stderr}");
println!("Exercise 6.1 passed");


## 8. Capstone: the Three-Macro Build

`TRIO_SRC` defines three macros — `tally!`, `pair!`, `shout!` — and its
`main` is stubbed. Fill the body so the program prints, one per line, in
order:

1. `tally=24` (from `tally![7, 8, 9]`)
2. `pair=BUY x5` (from `pair!(buy 5)`)
3. `shout=HELLO` (from `shout!(hello)`)


In [ ]:
// 8.1 Exercise: fill the stubbed main so all three receipts print.
const TRIO_SRC: &str = r#"macro_rules! tally {
    ($($n:expr),* $(,)?) => {{
        #[allow(unused_mut)]
        let mut acc = 0_i64;
        $( acc += $n; )*
        acc
    }};
}

macro_rules! pair {
    (buy $qty:expr) => {
        format!("BUY x{}", $qty)
    };
    (sell $qty:expr) => {
        format!("SELL x{}", $qty)
    };
}

macro_rules! shout {
    ($w:ident) => {
        stringify!($w).to_uppercase()
    };
}

fn main() {
    todo!()
}
"#;

// --- check (do not modify) ---
let (out, code) = desk_run("trio", TRIO_SRC, &[]);
println!("{out}");
assert!(out.contains("tally=24"), "out: {out}");
assert!(out.contains("pair=BUY x5"), "out: {out}");
assert!(out.contains("shout=HELLO"), "out: {out}");
assert_eq!(code, 0);
println!("Exercise 8.1 passed");


### What Comes Next

You have now written the machinery behind every receipt in this course.
`assert_eq!` was `macro_rules!`; `#[test]` was an attribute macro; the desk
you drove compiles your suites exactly the way `cargo test` does. Where you
take it next — `lazy_static!`, builder derives, your own DSLs — the shapes in
this chapter are the whole grammar.
